In [1]:
import pandas as pd

df_k = pd.read_csv('kotra_cleaned.csv', encoding='utf-8-sig')
df_c = pd.read_csv('customs_cleaned.csv', encoding='utf-8-sig')

### 0. 전제 확인
- KOTRA:    imposing_country (ISO코드) × hs_4digit × start_ym
- 관세청:   country (ISO코드)          × hs_code   × year_month
- 병합 키:  country = imposing_country
            hs_code = hs_4digit
            year_month = start_ym 기준 T-1 ~ T-12

### 1. 데이터 타입 통일

In [2]:
df_k['hs_4digit']  = df_k['hs_4digit'].astype(str)
df_c['hs_code']    = df_c['hs_code'].astype(str)
df_k['start_ym']   = pd.PeriodIndex(df_k['start_ym'].astype(str), freq='M')
df_c['year_month'] = pd.PeriodIndex(df_c['year_month'].astype(str), freq='M')

### 2. Kotra 필터링
- 관세청 데이터에 있는 국가만 포함
- 관세청 데이터에 있는 날짜만 포함

In [5]:
df_c['country'].unique()

array(['CA', 'CN', 'EU', 'IN', 'TH', 'GB', 'US'], dtype=object)

In [6]:
df_k['imposing_country'].unique()

array(['AM', 'EU', 'AU', 'BR', 'BY', 'CA', 'CN', 'CO', 'DO', 'EG', 'GB',
       'ID', 'IN', 'JP', 'KG', 'KZ', 'MA', 'MX', 'MY', 'NZ', 'PK', 'RU',
       'TH', 'TR', 'TW', 'UA', 'US', 'VN', 'ZA'], dtype=object)

In [7]:
available_countries = ['US', 'CA', 'GB', 'TH', 'CN', 'IN', 'EU']
df_k_filtered = df_k[
    (df_k['imposing_country'].isin(available_countries)) &
    (df_k['start_ym'] >= pd.Period('2010-01', freq='M'))
].copy()
print(f"병합 대상 KOTRA: {len(df_k_filtered)}행")

병합 대상 KOTRA: 5486행


### 3. 윈도우 병합

In [8]:
rows = []

for _, kotra_row in df_k_filtered.iterrows():
    country = kotra_row['imposing_country']
    hs      = kotra_row['hs_4digit']
    t0      = kotra_row['start_ym']

    window = df_c[
        (df_c['country']    == country) &
        (df_c['hs_code']    == hs) &
        (df_c['year_month'] >= t0 - 12) &
        (df_c['year_month'] <  t0)
    ].copy()

    if len(window) == 0:
        continue

    window['months_before'] = window['year_month'].apply(
        lambda x: (t0 - x).n
    )

    for col in ['case_id', 'imposing_country', 'hs_4digit',
                'target_country', 'start_ym', 'regulation_type',
                'regulation_status', 'is_korea_target',
                'tariff_clean', 'date_type', 'start_date', 'end_date']:
        window[col] = kotra_row[col]

    rows.append(window)

df_merged = pd.concat(rows, ignore_index=True)

### 4. 변화율 재계산
- case_id 별로 특정 시점의 평균 단가와 평균 물량을 피벗 테이블로 만듦
- 기준 시점(1개월 전) 대비 과거 3, 6, 12개월 전과의 가격/물량 변화율 계산

In [11]:
# Step 1. 필요한 시점 피벗
key_months = [1, 3, 6, 12]
df_key = df_merged[df_merged['months_before'].isin(key_months)].copy()

pivot_price = df_key.pivot_table(
    index='case_id',
    columns='months_before',
    values='unit_price',
    aggfunc='mean'
)
pivot_vol = df_key.pivot_table(
    index='case_id',
    columns='months_before',
    values='export_weight',
    aggfunc='mean'
)

pivot_price.columns = [f'price_t{c}' for c in pivot_price.columns]
pivot_vol.columns   = [f'vol_t{c}'   for c in pivot_vol.columns]

# Step 2. 변화율 계산
pivot = pd.concat([pivot_price, pivot_vol], axis=1).reset_index()

pivot['price_chg_3m_new']  = (pivot['price_t1'] / pivot['price_t3']  - 1) * 100
pivot['price_chg_6m_new']  = (pivot['price_t1'] / pivot['price_t6']  - 1) * 100
pivot['price_chg_12m_new'] = (pivot['price_t1'] / pivot['price_t12'] - 1) * 100
pivot['vol_chg_3m_new']    = (pivot['vol_t1'] / pivot['vol_t3']  - 1) * 100
pivot['vol_chg_6m_new']    = (pivot['vol_t1'] / pivot['vol_t6']  - 1) * 100
pivot['vol_chg_12m_new']   = (pivot['vol_t1'] / pivot['vol_t12'] - 1) * 100

# Step 3. 원본에 붙이고 기존 변화율 제거
df_merged = df_merged.merge(
    pivot[['case_id',
           'price_chg_3m_new', 'price_chg_6m_new', 'price_chg_12m_new',
           'vol_chg_3m_new',   'vol_chg_6m_new',   'vol_chg_12m_new']],
    on='case_id',
    how='left'
)

df_merged = df_merged.drop(columns=[
    'price_chg_3m', 'vol_chg_3m',
    'price_chg_6m', 'vol_chg_6m',
    'price_chg_12m','vol_chg_12m'
])

print(f"최종 행 수: {len(df_merged)}")
print(f"컬럼: {df_merged.columns.tolist()}")
print(f"\n새 변화율 결측값:")
new_chg_cols = ['price_chg_3m_new','price_chg_6m_new','price_chg_12m_new',
                'vol_chg_3m_new',  'vol_chg_6m_new',  'vol_chg_12m_new']
print(df_merged[new_chg_cols].isnull().sum())

최종 행 수: 52358
컬럼: ['country', 'hs_code', 'export_value', 'export_weight', 'unit_price', 'year_month', 'months_before', 'case_id', 'imposing_country', 'hs_4digit', 'target_country', 'start_ym', 'regulation_type', 'regulation_status', 'is_korea_target', 'tariff_clean', 'date_type', 'start_date', 'end_date', 'price_chg_3m_new', 'price_chg_6m_new', 'price_chg_12m_new', 'vol_chg_3m_new', 'vol_chg_6m_new', 'vol_chg_12m_new']

새 변화율 결측값:
price_chg_3m_new     3064
price_chg_6m_new     2228
price_chg_12m_new    2506
vol_chg_3m_new       3064
vol_chg_6m_new       2228
vol_chg_12m_new      2506
dtype: int64


### 4. 검증
- Kotra 데이터 수가 약간 아쉽기는 한데 더 없으니까 어쩔 수 없음

In [12]:
print(f"\n병합 후 전체: {len(df_merged)}행")
print(f"KOTRA 사례 수: {df_merged['case_id'].nunique()}건")
print(f"\nis_korea_target 분포:")
print(df_merged['is_korea_target'].value_counts())
print(f"클래스 비율 1:{df_merged['is_korea_target'].eq(0).sum()/df_merged['is_korea_target'].sum():.1f}")
print(f"\nmonths_before 분포:")
print(df_merged['months_before'].value_counts().sort_index())
print(f"\n국가별 행 수:")
print(df_merged['imposing_country'].value_counts())
print(f"\n결측값:")
print(df_merged.isnull().sum())


병합 후 전체: 52358행
KOTRA 사례 수: 112건

is_korea_target 분포:
is_korea_target
1    26654
0    25704
Name: count, dtype: int64
클래스 비율 1:1.0

months_before 분포:
months_before
1     4134
2     4237
3     4292
4     4375
5     4421
6     4432
7     4241
8     4327
9     4449
10    4549
11    4471
12    4430
Name: count, dtype: int64

국가별 행 수:
imposing_country
US    23998
CA     9908
EU     6732
TH     5763
GB     4573
CN     1024
IN      360
Name: count, dtype: int64

결측값:
country                 0
hs_code                 0
export_value            0
export_weight           0
unit_price              0
year_month              0
months_before           0
case_id                 0
imposing_country        0
hs_4digit               0
target_country          0
start_ym                0
regulation_type         0
regulation_status       0
is_korea_target         0
tariff_clean         1453
date_type               0
start_date              0
end_date             8321
price_chg_3m_new     3064
price_chg_6m_n

In [13]:
df_merged.to_csv('merged.csv', index=False, encoding='utf-8-sig')
print("\n✅ merged.csv 저장 완료")


✅ merged.csv 저장 완료
